In [ ]:
# ============================================================================
# CAD U-Net 3D for Top Brain CT Segmentation - Complete Colab Pipeline
# ============================================================================

#@title Setup & Imports
# If using Colab, uncomment the lines below:
!pip install -q torchio nibabel tqdm matplotlib
!pip install -q scikit-image
# from google.colab import drive
# drive.mount('/content/drive')

import os, glob, random, math
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import nibabel as nib
from tqdm import tqdm

try:
    import torchio as tio
except Exception:
    print("TorchIO not available. Install with `pip install torchio` for 3D augmentations.")
    tio = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.0/194.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 19.9 MB/s eta 0:00:00
Device: cuda


In [ ]:
#@title Configuration
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/nnUNet_raw/Dataset501_TopBrain_Segmentation"
IMG_GLOB = "imagesTr_topbrain_ct/*.nii.gz"
MSK_GLOB = "labelsTr_topbrain_ct/*.nii.gz"
LABELMAP_TXT = "/content/drive/MyDrive/Colab Notebooks/nnUNet_raw/Dataset501_TopBrain_Segmentation/labelmap_topbrain_ct.txt"

# Classes
NUM_CLASSES = 41   # 0..40 (0 = background)
BG_INDEX = 0

# Model & training
PATCH_SIZE = (64, 128, 128)  # (D, H, W) for 3D patches
BASE_CH = 32                  # Base channels for CAD U-Net 3D
BATCH_SIZE = 2                # Small batch size for 3D (memory intensive)
EPOCHS = 80
LR_MAX = 1e-3                 # OneCycle peak LR
WEIGHT_DECAY = 1e-5
ACCUM_STEPS = 2               # Gradient accumulation steps

# Loss weights - OPTIMIZED FOR VESSELS
ALPHA_CE = 0.8      # Slightly reduced
BETA_DICE = 1.0     # Standard
GAMMA_SKEL = 2.0    # ⭐ DOUBLED for vessel connectivity!
DO_TUBE = True
NUM_DILATIONS = 3 # ⭐ INCREASED for better vessel coverage

# Sampling
FOREGROUND_SAMPLING_PROB = 0.5
TRAIN_VAL_SPLIT = 0.8         # 80% train, 20% validation

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

# Results
RESULTS_DIR = "/content/drive/MyDrive/3D_CAD_Unet/Skeleton_loss/Results"
Checkpoint = "/content/drive/MyDrive/3D_CAD_Unet/Skeleton_loss/Model"
CHECKPOINT_PATH = os.path.join(Checkpoint, "best_CADUNet3D_SL.pt")
print("Saving results to:", RESULTS_DIR)


Saving results to: /content/drive/MyDrive/3D_CAD_Unet/Skeleton_loss/Results


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#@title Labelmap Reader + Label Decoding

def read_itksnap_labelmap_txt(path):
    """Read ITK-SNAP labelmap text file."""
    labels = {}
    if not os.path.exists(path):
        print("Labelmap not found:", path, "— proceeding without RGB mapping.")
        return labels
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            try:
                idx = int(parts[0])
            except Exception:
                continue
            entry = {"idx": idx, "name": None, "rgb": None}
            if len(parts) >= 7:
                try:
                    r, g, b = int(parts[3]), int(parts[4]), int(parts[5])
                    entry["rgb"] = (r, g, b)
                except Exception:
                    pass
            labels[idx] = entry
    print(f"Loaded {len(labels)} classes from labelmap file.")
    return labels

LABEL_INFO = read_itksnap_labelmap_txt(LABELMAP_TXT)

def palette_png_to_indexed(mask_img, palette_rgb_to_idx):
    """Convert RGB palette mask to indexed mask."""
    m = np.array(mask_img)
    if m.ndim == 2:
        return m.astype(np.int64)
    if m.ndim == 3 and m.shape[2] == 4:
        m = m[..., :3]
    h, w, _ = m.shape
    out = np.zeros((h * w,), dtype=np.int64)
    flat = m.reshape(-1, 3)
    for i in range(flat.shape[0]):
        col = tuple(int(v) for v in flat[i])
        out[i] = palette_rgb_to_idx.get(col, BG_INDEX)
    return out.reshape(h, w)

def ensure_indexed_mask(mask_array, num_classes=NUM_CLASSES):
    """Ensure mask is indexed format (not RGB)."""
    m = np.array(mask_array)
    if m.ndim == 3 and m.shape[-1] in (3, 4):
        rgb2idx = {}
        for entry in LABEL_INFO.values():
            if entry.get("rgb") is not None:
                rgb2idx[entry["rgb"]] = entry["idx"]
        if not rgb2idx:
            raise ValueError("RGB mask provided but labelmap has no RGB entries.")
        m = palette_png_to_indexed(m, rgb2idx)
    m = m.astype(np.int64)
    m[(m < 0) | (m >= num_classes)] = BG_INDEX
    return m


Loaded 41 classes from labelmap file.


In [ ]:
#@title CAD U-Net 3D Architecture

class SEBlock3D(nn.Module):
    """Squeeze-and-Excitation block for 3D."""
    def __init__(self, ch, r=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Conv3d(ch, ch // r, 1),
            nn.ReLU(inplace=True),
            nn.Conv3d(ch // r, ch, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.fc(self.pool(x))
        return x * w

class SpatialAttn3D(nn.Module):
    """Spatial attention for 3D."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv3d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        attn = self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn

def Norm3d(C):
    """Instance normalization for 3D (works better with small batches)."""
    return nn.InstanceNorm3d(C, affine=True, eps=1e-5)

class CADBlock3D(nn.Module):
    """Context-Aware Dilated block with parallel dilated convolutions."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 2

        self.d1 = nn.Sequential(
            nn.Conv3d(in_ch, mid, 3, padding=1, dilation=1),
            Norm3d(mid), nn.ReLU(inplace=True)
        )
        self.d2 = nn.Sequential(
            nn.Conv3d(in_ch, mid // 2, 3, padding=2, dilation=2),
            Norm3d(mid // 2), nn.ReLU(inplace=True)
        )
        self.d3 = nn.Sequential(
            nn.Conv3d(in_ch, mid // 2, 3, padding=4, dilation=4),
            Norm3d(mid // 2), nn.ReLU(inplace=True)
        )
        self.mix = nn.Sequential(
            nn.Conv3d(mid + mid // 2 + mid // 2, out_ch, 1),
            Norm3d(out_ch), nn.ReLU(inplace=True)
        )
        self.se = SEBlock3D(out_ch)
        self.sa = SpatialAttn3D()

    def forward(self, x):
        x1 = self.d1(x)
        x2 = self.d2(x)
        x3 = self.d3(x)
        z = self.mix(torch.cat([x1, x2, x3], dim=1))
        z = self.se(z)
        z = self.sa(z)
        return z

class AttnGate3D(nn.Module):
    """Attention gate for skip connections."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv3d(F_g, F_int, 1), Norm3d(F_int))
        self.W_x = nn.Sequential(nn.Conv3d(F_l, F_int, 1), Norm3d(F_int))
        self.psi = nn.Sequential(nn.Conv3d(F_int, 1, 1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class Up3D(nn.Module):
    """Upsampling block with attention gate."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.attn = AttnGate3D(F_g=out_ch, F_l=skip_ch, F_int=out_ch // 2)
        self.block = nn.Sequential(
            CADBlock3D(out_ch + skip_ch, out_ch),
            CADBlock3D(out_ch, out_ch)
        )

    def forward(self, x, skip):
        x = self.up(x)
        # Handle size mismatch
        dz = skip.size(2) - x.size(2)
        dy = skip.size(3) - x.size(3)
        dx = skip.size(4) - x.size(4)
        if dz or dy or dx:
            x = F.pad(x, (0, max(0, dx), 0, max(0, dy), 0, max(0, dz)))
        skip = self.attn(x, skip)
        x = torch.cat([x, skip], dim=1)
        x = self.block(x)
        return x

class CADUNet3D(nn.Module):
    """CAD U-Net 3D with attention gates and context-aware blocks."""
    def __init__(self, in_ch=1, base=32, num_classes=41):
        super().__init__()
        # Encoder
        self.e1 = nn.Sequential(CADBlock3D(in_ch, base), CADBlock3D(base, base))
        self.p1 = nn.MaxPool3d(2)
        self.e2 = nn.Sequential(CADBlock3D(base, base*2), CADBlock3D(base*2, base*2))
        self.p2 = nn.MaxPool3d(2)
        self.e3 = nn.Sequential(CADBlock3D(base*2, base*4), CADBlock3D(base*4, base*4))
        self.p3 = nn.MaxPool3d(2)
        self.e4 = nn.Sequential(CADBlock3D(base*4, base*8), CADBlock3D(base*8, base*8))
        self.p4 = nn.MaxPool3d(2)

        # Bottleneck
        self.bott = nn.Sequential(CADBlock3D(base*8, base*16), CADBlock3D(base*16, base*16))

        # Decoder
        self.u4 = Up3D(base*16, base*8, base*8)
        self.u3 = Up3D(base*8, base*4, base*4)
        self.u2 = Up3D(base*4, base*2, base*2)
        self.u1 = Up3D(base*2, base, base)

        # Output head
        self.head_mc = nn.Conv3d(base, num_classes, 1)

    def forward(self, x):
        c1 = self.e1(x)
        p1 = self.p1(c1)
        c2 = self.e2(p1)
        p2 = self.p2(c2)
        c3 = self.e3(p2)
        p3 = self.p3(c3)
        c4 = self.e4(p3)
        p4 = self.p4(c4)

        b = self.bott(p4)

        x = self.u4(b, c4)
        x = self.u3(x, c3)
        x = self.u2(x, c2)
        x = self.u1(x, c1)

        return self.head_mc(x)

# Test model
print("\n" + "="*60)
print("Testing CAD U-Net 3D architecture...")
print("="*60)
with torch.no_grad():
    test_model = CADUNet3D(in_ch=1, base=BASE_CH, num_classes=NUM_CLASSES).to(device)
    test_input = torch.randn(1, 1, 32, 64, 64, device=device)
    test_output = test_model(test_input)
    print(f"✓ Model initialized successfully")
    print(f"  Input shape:  {tuple(test_input.shape)}")
    print(f"  Output shape: {tuple(test_output.shape)}")
    print(f"  Parameters: {sum(p.numel() for p in test_model.parameters()):,}")
    del test_model, test_input, test_output
    torch.cuda.empty_cache()


Testing CAD U-Net 3D architecture...
✓ Model initialized successfully
  Input shape:  (1, 1, 32, 64, 64)
  Output shape: (1, 41, 32, 64, 64)
  Parameters: 23,677,091


In [ ]:
print("\n[1/5] Installing dependencies...")
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-image"])
print("✓ scikit-image installed")


[1/5] Installing dependencies...
✓ scikit-image installed


In [ ]:
print("\n[2/5] Importing modules...")
from skimage.morphology import skeletonize, dilation, binary_dilation
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Callable
print("✓ All modules imported")


[2/5] Importing modules...
✓ All modules imported


In [ ]:
# Replace your skeleton function with:
def generate_skeleton_transform(segmentation, do_tube=True, num_dilations=2):
    from skimage.morphology import skeletonize, dilation, ball

    bin_seg = (segmentation > 0)
    if not np.sum(bin_seg) > 0:
        return np.zeros_like(segmentation, dtype=np.int16)

    try:
        # REMOVED: binary_closing line
        skel = skeletonize(bin_seg)
        skel = (skel > 0).astype(np.int16)

        if do_tube:
            struct_elem = ball(1)
            for _ in range(num_dilations):
                skel = dilation(skel, footprint=struct_elem)

        skel = skel * segmentation.astype(np.int16)
        return skel

    except Exception as e:
        return segmentation.astype(np.int16)

In [ ]:
#@title Loss Functions

class DiceLossMultiClass(nn.Module):
    """Multi-class Dice loss."""
    def __init__(self, smooth=1.0, eps=1e-7, weight=None, ignore_index=None, reduction="mean"):
        super().__init__()
        self.smooth = smooth
        self.eps = eps
        self.register_buffer("weight", weight if weight is not None else None)
        self.ignore_index = ignore_index
        assert reduction in ("mean", "sum", "none")
        self.reduction = reduction

    def forward(self, logits, target):
        B, C = logits.shape[:2]
        probs = F.softmax(logits, dim=1)

        with torch.no_grad():
            target_oh = F.one_hot(target.long(), num_classes=C)
            dims = list(range(1, target_oh.dim()-1))
            target_oh = target_oh.permute(0, -1, *dims).float()

            if self.ignore_index is not None:
                valid = (target != self.ignore_index).unsqueeze(1).float()
                target_oh = target_oh * valid
                probs = probs * valid

        probs_f = probs.reshape(B, C, -1)
        tgt_f = target_oh.reshape(B, C, -1)

        inter = (probs_f * tgt_f).sum(-1).sum(0)
        card = (probs_f + tgt_f).sum(-1).sum(0)

        dice = (2 * inter + self.smooth) / (card + self.smooth + self.eps)
        loss_c = 1.0 - dice

        if self.weight is not None:
            if self.weight.numel() != logits.size(1):
                raise ValueError("Dice weight length must equal num classes")
            loss_c = loss_c * self.weight

        if self.reduction == "mean":
            return loss_c.mean()
        if self.reduction == "sum":
            return loss_c.sum()
        return loss_c

class CombinedLossMultiClass(nn.Module):
    """Combined Cross-Entropy + Dice loss."""
    def __init__(self, alpha=1.0, beta=1.0, dice_kwargs=None, ce_weight=None, ignore_index=None):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

        # CrossEntropyLoss requires ignore_index to be int, not None
        if ignore_index is None:
            self.ce = nn.CrossEntropyLoss(weight=ce_weight)
        else:
            self.ce = nn.CrossEntropyLoss(weight=ce_weight, ignore_index=ignore_index)

        dice_kwargs = dice_kwargs or {}
        if 'ignore_index' not in dice_kwargs:
            dice_kwargs['ignore_index'] = ignore_index
        self.dice = DiceLossMultiClass(**dice_kwargs)

    def forward(self, logits, target):
        return self.alpha * self.ce(logits, target.long()) + self.beta * self.dice(logits, target)

print("✓ Loss functions defined")

✓ Loss functions defined


In [ ]:
#@title Skeleton Recall Loss Functions
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Callable

print("\n[4/5] Defining loss functions...")

class SoftSkeletonRecallLoss(nn.Module):
    """
    Skeleton Recall Loss - measures how well predictions cover skeleton structures.
    Based on the implementation in dice (2).py
    """
    def __init__(self, apply_nonlin: Callable = None, batch_dice: bool = False,
                 smooth: float = 1.0, do_bg: bool = False):
        super(SoftSkeletonRecallLoss, self).__init__()

        if do_bg:
            raise RuntimeError("Skeleton recall does not work with background class")

        self.batch_dice = batch_dice
        self.apply_nonlin = apply_nonlin
        self.smooth = smooth

    def forward(self, x, y_skel, loss_mask=None):
        """
        Args:
            x: Network output logits [B, C, D, H, W]
            y_skel: Tubed skeleton ground truth [B, D, H, W]
            loss_mask: Optional mask for valid regions
        """
        shp_x, shp_y = x.shape, y_skel.shape

        # Apply softmax
        if self.apply_nonlin is not None:
            x = self.apply_nonlin(x)

        # Remove background channel
        x = x[:, 1:]

        # Spatial dimensions for summation
        axes = list(range(2, len(shp_x)))

        with torch.no_grad():
            # Ensure correct shape
            if len(shp_x) != len(shp_y):
                y_skel = y_skel.view((shp_y[0], 1, *shp_y[1:]))

            # Convert to one-hot
            if all([i == j for i, j in zip(shp_x, shp_y)]):
                y_onehot = y_skel[:, 1:]
            else:
                gt = y_skel.long()
                y_onehot = torch.zeros(shp_x, device=x.device, dtype=y_skel.dtype)
                y_onehot.scatter_(1, gt, 1)
                y_onehot = y_onehot[:, 1:]

            sum_gt = y_onehot.sum(axes) if loss_mask is None else (y_onehot * loss_mask).sum(axes)

        # Compute intersection
        inter_rec = (x * y_onehot).sum(axes) if loss_mask is None else (x * y_onehot * loss_mask).sum(axes)

        if self.batch_dice:
            inter_rec = inter_rec.sum(0)
            sum_gt = sum_gt.sum(0)

        # Compute recall
        rec = (inter_rec + self.smooth) / (torch.clip(sum_gt + self.smooth, 1e-8))
        rec = rec.mean()

        return -rec


class CombinedLossWithSkeletonRecall(nn.Module):
    """Vessel-optimized with class weighting"""
    def __init__(self, alpha_ce=0.8, beta_dice=1.0, gamma_skel=2.0,
                 num_classes=41, class_weights=None, ignore_index=None):
        super().__init__()

        self.alpha = alpha_ce
        self.beta = beta_dice
        self.gamma = gamma_skel

        # ADDED: Class weighting support
        if ignore_index is not None:
            self.ce = nn.CrossEntropyLoss(weight=class_weights, ignore_index=ignore_index)
        else:
            self.ce = nn.CrossEntropyLoss(weight=class_weights)

        self.dice = self._dice_loss

        self.skel_recall = SoftSkeletonRecallLoss(
            apply_nonlin=lambda x: F.softmax(x, dim=1),
            batch_dice=False,
            smooth=1.0,
            do_bg=False
        )

    def _dice_loss(self, logits, target):
        """Simple Dice loss implementation"""
        probs = F.softmax(logits, dim=1)
        num_classes = logits.shape[1]

        # One-hot encode target
        target_oh = F.one_hot(target.long(), num_classes=num_classes)
        target_oh = target_oh.permute(0, -1, *range(1, target_oh.dim()-1)).float()

        # Flatten
        probs_flat = probs.reshape(probs.shape[0], probs.shape[1], -1)
        target_flat = target_oh.reshape(target_oh.shape[0], target_oh.shape[1], -1)

        # Dice per class
        intersection = (probs_flat * target_flat).sum(-1)
        cardinality = probs_flat.sum(-1) + target_flat.sum(-1)

        dice = (2.0 * intersection + 1.0) / (cardinality + 1.0)
        dice = dice[:, 1:].mean()  # Exclude background

        return 1.0 - dice

    def forward(self, logits, target, skel):
        """
        Args:
            logits: [B, C, D, H, W]
            target: [B, D, H, W]
            skel: [B, D, H, W]
        """
        ce_loss = self.alpha * self.ce(logits, target.long())
        dice_loss = self.beta * self.dice(logits, target)
        skel_loss = self.gamma * self.skel_recall(logits, skel)

        return ce_loss + dice_loss + skel_loss

print("✓ Loss functions defined")



[4/5] Defining loss functions...
✓ Loss functions defined


In [ ]:
#@title Updated Dataset Class with Skeleton Generation
class TopBrainCT3DWithSkeleton(Dataset):
    """Dataset with automatic skeleton generation"""
    def __init__(self, img_files, msk_files,
                 patch_size=(64, 128, 128),
                 augment=True,
                 foreground_sampling=True,
                 num_classes=41,
                 cache_volumes=True,
                 patches_per_volume=10,
                 do_tube=True,
                 num_dilations=2):

        assert len(img_files) == len(msk_files)
        self.img_files = img_files
        self.msk_files = msk_files
        self.patch_size = patch_size
        self.augment = augment
        self.fg_sampling = foreground_sampling
        self.num_classes = num_classes
        self.cache_volumes = cache_volumes
        self.patches_per_volume = patches_per_volume
        self.do_tube = do_tube
        self.num_dilations = num_dilations
        self._cache = {}

        # Augmentation (requires torchio)
        try:
            import torchio as tio
            if self.augment:
                self.transforms = tio.Compose([
                    tio.RandomFlip(axes=('LR',), flip_probability=0.5),
                    tio.RandomAffine(scales=(0.9, 1.1), degrees=5, isotropic=True, p=0.5),
                    tio.RandomElasticDeformation(num_control_points=7, max_displacement=5, p=0.3),
                    tio.RandomGamma(log_gamma=(-0.3, 0.3), p=0.3),
                ])
            else:
                self.transforms = None
        except:
            self.transforms = None
            print("⚠ TorchIO not available - augmentation disabled")

    def _load_volume(self, idx):
        if self.cache_volumes and idx in self._cache:
            return self._cache[idx]

        import nibabel as nib
        vol = nib.load(self.img_files[idx]).get_fdata().astype(np.float32)
        msk = nib.load(self.msk_files[idx]).get_fdata().astype(np.int16)

        # Normalize
        vol = (vol - vol.mean()) / (vol.std() + 1e-6)

        # Ensure indexed mask (assumes ensure_indexed_mask is defined)
        msk = ensure_indexed_mask(msk, self.num_classes)

        # Generate skeleton
        skel = generate_skeleton_transform(msk, self.do_tube, self.num_dilations)

        if self.cache_volumes:
            self._cache[idx] = (vol, msk, skel)

        return vol, msk, skel

    def _sample_patch(self, vol, msk, skel):
        D, H, W = vol.shape
        pd, ph, pw = self.patch_size

        import random
        attempts = 5 if self.fg_sampling else 1
        for _ in range(attempts):
            dz = random.randint(0, max(0, D - pd))
            dy = random.randint(0, max(0, H - ph))
            dx = random.randint(0, max(0, W - pw))

            sub_msk = msk[dz:dz+pd, dy:dy+ph, dx:dx+pw]
            if not self.fg_sampling or np.any(sub_msk > 0):
                break

        return (vol[dz:dz+pd, dy:dy+ph, dx:dx+pw],
                msk[dz:dz+pd, dy:dy+ph, dx:dx+pw],
                skel[dz:dz+pd, dy:dy+ph, dx:dx+pw])

    def __len__(self):
        return len(self.img_files) * self.patches_per_volume

    def __getitem__(self, idx):
        vidx = idx % len(self.img_files)
        vol, msk, skel = self._load_volume(vidx)
        v, m, s = self._sample_patch(vol, msk, skel)

        # Apply augmentations
        if self.transforms is not None:
            import torchio as tio
            v_tensor = torch.from_numpy(v).float().unsqueeze(0)
            m_tensor = torch.from_numpy(m).long().unsqueeze(0)
            s_tensor = torch.from_numpy(s).long().unsqueeze(0)

            subj = tio.Subject(
                image=tio.ScalarImage(tensor=v_tensor),
                mask=tio.LabelMap(tensor=m_tensor),
                skeleton=tio.LabelMap(tensor=s_tensor)
            )
            out = self.transforms(subj)

            v = out.image.data[0].numpy()
            m = out.mask.data[0].numpy()
            s = out.skeleton.data[0].numpy()

        img = torch.from_numpy(v).float().unsqueeze(0)
        mask = torch.from_numpy(m).long()
        skeleton = torch.from_numpy(s).long()

        return img, mask, skeleton

print("✓ Dataset class defined")

✓ Dataset class defined


In [ ]:
#@title Validation Function
@torch.no_grad()
def validate_with_skeleton(model, loader, device, num_classes=41):
    """Validate with both Dice and Skeleton Recall metrics."""
    model.eval()
    dices = torch.zeros(num_classes, device=device)
    skel_recalls = torch.zeros(num_classes, device=device)
    counts = torch.zeros(num_classes, device=device)

    for imgs, masks, skels in tqdm(loader, desc="Validation", leave=False):
        imgs = imgs.to(device)
        masks = masks.to(device)
        skels = skels.to(device)

        logits = model(imgs)
        preds = torch.argmax(logits, dim=1)
        probs = F.softmax(logits, dim=1)

        for c in range(num_classes):
            # Regular Dice on full mask
            p = (preds == c).float()
            t = (masks == c).float()
            inter = (p * t).sum()
            den = p.sum() + t.sum()
            if den > 0:
                dices[c] += (2 * inter / den)
                counts[c] += 1

            # Skeleton recall
            t_skel = (skels == c).float()
            inter_skel = (probs[:, c] * t_skel).sum()
            den_skel = t_skel.sum()
            if den_skel > 0:
                skel_recalls[c] += (inter_skel / den_skel)

    mean_dice = (dices[counts > 0] / counts[counts > 0]).mean().item()
    mean_skel_recall = (skel_recalls[counts > 0] / counts[counts > 0]).mean().item()

    return mean_dice, mean_skel_recall

In [ ]:
#@title Updated Training Function with Skeleton Loss

def train_one_epoch_with_skeleton(model, loader, optimizer, criterion, device, accum_steps=1):
    """Training loop with skeleton loss"""
    from torch.cuda.amp import autocast, GradScaler
    from tqdm import tqdm

    model.train()
    scaler = GradScaler()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc="Training", leave=False)
    for batch_idx, (imgs, masks, skels) in enumerate(pbar):
        imgs = imgs.to(device)
        masks = masks.to(device)
        skels = skels.to(device)

        with autocast():
            logits = model(imgs)
            loss = criterion(logits, masks, skels) / accum_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * accum_steps
        pbar.set_postfix({'loss': f'{loss.item() * accum_steps:.4f}'})

    return total_loss / max(1, len(loader))


def fit_with_skeleton(model, train_loader, val_loader, epochs=50, lr=1e-4,
                      weight_decay=1e-5, num_classes=41, save_path="best_model.pt",
                      accum_steps=1, alpha_ce=1.0, beta_dice=1.0, gamma_skel=0.5):
    """Complete training loop with skeleton recall loss"""

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Create criterion with class weights
    criterion = CombinedLossWithSkeletonRecall(
        alpha_ce=ALPHA_CE,
        beta_dice=BETA_DICE,
        gamma_skel=GAMMA_SKEL,
        num_classes=NUM_CLASSES,
        class_weights=class_weights  # ⭐ NEW
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
        anneal_strategy='cos'
    )

    best = -1
    history = {'train_loss': [], 'val_dice': [], 'lr': []}

    print("\n" + "="*70)
    print("Training with Skeleton Recall Loss")
    print("="*70)
    print(f"Loss weights: CE={alpha_ce}, Dice={beta_dice}, Skeleton={gamma_skel}")
    print("="*70)

    for ep in range(1, epochs + 1):
        tr_loss = train_one_epoch_with_skeleton(
            model, train_loader, optimizer, criterion, device, accum_steps
        )

        # Use your existing validate function
        val_dice = validate_with_skeleton(model, val_loader, device, num_classes=num_classes)

        history['train_loss'].append(tr_loss)
        history['val_dice'].append(val_dice)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        print(f"Epoch {ep:03d}/{epochs} | train_loss={tr_loss:.4f} | "
              f"val_dice={val_dice:.4f} | lr={optimizer.param_groups[0]['lr']:.6f}")

        if val_dice > best:
            best = val_dice
            torch.save({
                'epoch': ep,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_dice': val_dice,
                'train_loss': tr_loss,
            }, save_path)
            print(f"  ✓ Saved best model (val_dice={best:.4f})")

        for _ in range(len(train_loader)):
            scheduler.step()

    print("\n" + "="*70)
    print(f"Training Complete! Best Val Dice: {best:.4f}")
    print("="*70)

    return history

print("✓ Training functions defined")

✓ Training functions defined


In [ ]:
#@title Skeleton Loss Configuration

# Loss weights (adjust these based on your needs)
ALPHA_CE = 1.0      # Cross-Entropy weight
BETA_DICE = 1.0     # Dice weight
GAMMA_SKEL = 0.5    # Skeleton Recall weight (start lower, can increase)

# Skeleton parameters
DO_TUBE = True      # Enable tubular dilation
TUBE_RADIUS = 2     # Dilation radius (pixels)

print("✓ Skeleton loss configuration set")
print(f"  CE weight: {ALPHA_CE}")
print(f"  Dice weight: {BETA_DICE}")
print(f"  Skeleton weight: {GAMMA_SKEL}")
print(f"  Tube dilation: {DO_TUBE} (radius={TUBE_RADIUS})")

✓ Skeleton loss configuration set
  CE weight: 1.0
  Dice weight: 1.0
  Skeleton weight: 0.5
  Tube dilation: True (radius=2)


In [ ]:
#@title Inference Functions

@torch.no_grad()
def sliding_window_predict(volume_np, model, patch_size=(64, 128, 128),
                          overlap=(0.5, 0.5, 0.5), num_classes=41):
    """Predict on full volume using sliding window."""
    model.eval()
    D, H, W = volume_np.shape
    pd, ph, pw = patch_size
    sd = max(1, int(pd * (1 - overlap[0])))
    sh = max(1, int(ph * (1 - overlap[1])))
    sw = max(1, int(pw * (1 - overlap[2])))

    out_logits = np.zeros((num_classes, D, H, W), dtype=np.float32)
    out_counts = np.zeros((D, H, W), dtype=np.float32)

    total_patches = (math.ceil(D/sd) * math.ceil(H/sh) * math.ceil(W/sw))
    pbar = tqdm(total=total_patches, desc="Predicting")

    for z in range(0, max(1, D - pd + 1), sd):
        for y in range(0, max(1, H - ph + 1), sh):
            for x in range(0, max(1, W - pw + 1), sw):
                patch = volume_np[z:z+pd, y:y+ph, x:x+pw]

                if patch.shape != (pd, ph, pw):
                    pad_d = pd - patch.shape[0]
                    pad_h = ph - patch.shape[1]
                    pad_w = pw - patch.shape[2]
                    patch = np.pad(patch, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='edge')

                tens = torch.from_numpy(patch).float().unsqueeze(0).unsqueeze(0).to(device)
                with autocast():
                    logits = model(tens)
                    logits = logits[0].cpu().float().numpy()

                dd, hh, ww = min(pd, D - z), min(ph, H - y), min(pw, W - x)
                out_logits[:, z:z+dd, y:y+hh, x:x+ww] += logits[:, :dd, :hh, :ww]
                out_counts[z:z+dd, y:y+hh, x:x+ww] += 1.0

                pbar.update(1)

    pbar.close()
    out_counts[out_counts == 0] = 1.0
    out_logits = out_logits / out_counts[None, ...]
    return out_logits

def save_nifti(mask_np, ref_img_path, out_path):
    """Save mask as NIfTI with reference header."""
    ref = nib.load(ref_img_path)
    nii = nib.Nifti1Image(mask_np.astype(np.int16), affine=ref.affine, header=ref.header)
    nib.save(nii, out_path)
    print(f"✓ Saved: {out_path}")

print("✓ Inference functions defined")

✓ Inference functions defined


In [ ]:
#@title Data Loading & Preparation

print("\n" + "="*60)
print("Loading Dataset")
print("="*60)

# Get file lists
img_pattern = os.path.join(DATA_DIR, IMG_GLOB)
msk_pattern = os.path.join(DATA_DIR, MSK_GLOB)

img_files = sorted(glob.glob(img_pattern))
msk_files = sorted(glob.glob(msk_pattern))

print(f"Found {len(img_files)} images and {len(msk_files)} masks")

if len(img_files) == 0:
    print("⚠ WARNING: No image files found! Check DATA_DIR and IMG_GLOB paths.")
    print(f"  Looking for: {img_pattern}")
else:
    print(f"✓ Sample image: {os.path.basename(img_files[0])}")

if len(msk_files) == 0:
    print("⚠ WARNING: No mask files found! Check DATA_DIR and MSK_GLOB paths.")
    print(f"  Looking for: {msk_pattern}")
else:
    print(f"✓ Sample mask: {os.path.basename(msk_files[0])}")

assert len(img_files) == len(msk_files), "Mismatch between images and masks count!"
assert len(img_files) > 0, "No data files found!"

# Train/val split
n_total = len(img_files)
n_train = int(n_total * TRAIN_VAL_SPLIT)

indices = list(range(n_total))
random.shuffle(indices)

train_indices = indices[:n_train]
val_indices = indices[n_train:]

train_imgs = [img_files[i] for i in train_indices]
train_msks = [msk_files[i] for i in train_indices]
val_imgs = [img_files[i] for i in val_indices]
val_msks = [msk_files[i] for i in val_indices]

print(f"\nSplit: {len(train_imgs)} training, {len(val_imgs)} validation")

# Create datasets
print("\nCreating datasets...")
# Create datasets with skeleton generation
# NEW:
train_ds = TopBrainCT3DWithSkeleton(
    train_imgs, train_msks,
    patch_size=PATCH_SIZE,
    augment=True,
    foreground_sampling=True,
    num_classes=NUM_CLASSES,
    cache_volumes=True,
    patches_per_volume=10,
    do_tube=DO_TUBE,           # ⭐ NEW
    num_dilations=NUM_DILATIONS # ⭐ NEW
)

val_ds = TopBrainCT3DWithSkeleton(
    val_imgs, val_msks,
    patch_size=PATCH_SIZE,
    augment=False,
    foreground_sampling=False,
    num_classes=NUM_CLASSES,
    cache_volumes=True,
    patches_per_volume=5,
    do_tube=DO_TUBE,           # ⭐ NEW
    num_dilations=NUM_DILATIONS # ⭐ NEW
)

print(f"✓ Training dataset: {len(train_ds)} patches")
print(f"✓ Validation dataset: {len(val_ds)} patches")

# Create dataloaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"✓ Training batches per epoch: {len(train_loader)}")
print(f"✓ Validation batches: {len(val_loader)}")


Loading Dataset
Found 40 images and 40 masks
✓ Sample image: topcow_ct_001_0000.nii.gz
✓ Sample mask: topcow_ct_001.nii.gz

Split: 32 training, 8 validation

Creating datasets...
✓ Training dataset: 320 patches
✓ Validation dataset: 40 patches
✓ Training batches per epoch: 160
✓ Validation batches: 20


In [ ]:
#@title Compute Class Weights for Vessels
from skimage.morphology import skeletonize, dilation, ball
from scipy.ndimage import binary_closing
import numpy as np

def compute_class_weights(train_ds, num_classes=41, device='cuda'):
    """Give higher weight to rare vessel classes"""
    print("\nComputing class weights...")

    class_counts = torch.zeros(num_classes, dtype=torch.float32)

    # Sample 5 volumes
    for idx in range(min(5, len(train_ds.img_files))):
        _, mask, _ = train_ds._load_volume(idx)
        for c in range(num_classes):
            class_counts[c] += (mask == c).sum()

    # Inverse frequency
    total = class_counts.sum()
    class_weights = total / (num_classes * class_counts + 1e-6)
    class_weights = class_weights / class_weights.mean()
    class_weights = torch.clamp(class_weights, 0.1, 10.0)
    class_weights[0] = 0.1  # Lower weight for background

    print(f"✓ Weights: bg={class_weights[0]:.2f}, "
          f"mean_fg={class_weights[1:].mean():.2f}, "
          f"max={class_weights.max():.2f}")

    return class_weights.to(device)

# Compute weights
class_weights = compute_class_weights(train_ds, NUM_CLASSES, device)


Computing class weights...
✓ Weights: bg=0.10, mean_fg=0.59, max=10.00


In [ ]:
#@title Initialize Model

print("\n" + "="*60)
print("Initializing Model")
print("="*60)

model = CADUNet3D(in_ch=1, base=BASE_CH, num_classes=NUM_CLASSES).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model: CAD U-Net 3D")
print(f"  Base channels: {BASE_CH}")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Device: {device}")


Initializing Model
✓ Model: CAD U-Net 3D
  Base channels: 32
  Number of classes: 41
  Total parameters: 23,677,091
  Trainable parameters: 23,677,091
  Device: cuda


In [ ]:
#@title ✅ Quick Verification - Run This Before Training

print("="*70)
print("Pre-Training Verification")
print("="*70)

# 1. Check class weights
print("\n1️⃣ Class Weights:")
print(f"   ✓ Background weight: {class_weights[0]:.2f}")
print(f"   ✓ Mean foreground: {class_weights[1:].mean():.2f}")
print(f"   ✓ Max (rare class): {class_weights.max():.2f}")
print(f"   Status: {'✓ GOOD' if class_weights.max() <= 10 else '⚠ Very high'}")

# 2. Test dataset loading with skeleton
print("\n2️⃣ Testing Dataset Loading...")
try:
    # Get one sample from training set
    img, mask, skel = train_ds[0]

    print(f"   ✓ Sample loaded successfully")
    print(f"   - Image shape: {img.shape}")
    print(f"   - Mask shape: {mask.shape}")
    print(f"   - Skeleton shape: {skel.shape}")

    # Check skeleton coverage
    fg_voxels = (mask > 0).sum().item()
    skel_voxels = (skel > 0).sum().item()

    if fg_voxels > 0:
        coverage = skel_voxels / fg_voxels * 100
        print(f"   - Foreground: {fg_voxels} voxels")
        print(f"   - Skeleton: {skel_voxels} voxels")
        print(f"   - Coverage: {coverage:.1f}%")

        if skel_voxels > 0:
            print(f"   Status: ✓ Skeleton generation WORKING")
        else:
            print(f"   Status: ✗ NO SKELETON - Check function!")
    else:
        print(f"   Status: ⚠ No foreground in this patch (try another)")

except Exception as e:
    print(f"   ✗ Error: {e}")
    import traceback
    traceback.print_exc()

# 3. Check loss configuration
print("\n3️⃣ Loss Configuration:")
print(f"   - ALPHA_CE (CrossEntropy): {ALPHA_CE}")
print(f"   - BETA_DICE (Dice): {BETA_DICE}")
print(f"   - GAMMA_SKEL (Skeleton): {GAMMA_SKEL} {'✓ HIGH (good for vessels)' if GAMMA_SKEL >= 1.5 else '⚠ LOW'}")
print(f"   - NUM_DILATIONS: {NUM_DILATIONS} {'✓ GOOD' if NUM_DILATIONS >= 3 else '⚠ Consider 3+'}")

# 4. Check model
print("\n4️⃣ Model Status:")
print(f"   - Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   - Device: {next(model.parameters()).device}")
print(f"   - Training mode: {model.training}")

# 5. Check data loaders
print("\n5️⃣ Data Loaders:")
print(f"   - Training batches: {len(train_loader)}")
print(f"   - Validation batches: {len(val_loader)}")
print(f"   - Batch size: {BATCH_SIZE}")
print(f"   - Accumulation steps: {ACCUM_STEPS}")
print(f"   - Effective batch: {BATCH_SIZE * ACCUM_STEPS}")

# 6. Test one forward pass
print("\n6️⃣ Testing Forward Pass...")
try:
    model.eval()
    with torch.no_grad():
        test_img = torch.randn(1, 1, 32, 64, 64).to(device)
        test_out = model(test_img)
        print(f"   ✓ Forward pass successful")
        print(f"   - Input: {test_img.shape}")
        print(f"   - Output: {test_out.shape}")
    del test_img, test_out
    torch.cuda.empty_cache()
except Exception as e:
    print(f"   ✗ Forward pass failed: {e}")

# 7. Test loss computation
print("\n7️⃣ Testing Loss Computation...")
try:
    criterion = CombinedLossWithSkeletonRecall(
        alpha_ce=ALPHA_CE,
        beta_dice=BETA_DICE,
        gamma_skel=GAMMA_SKEL,
        num_classes=NUM_CLASSES,
        class_weights=class_weights
    )

    # Get one batch
    for imgs, masks, skels in train_loader:
        imgs = imgs.to(device)
        masks = masks.to(device)
        skels = skels.to(device)

        with torch.no_grad():
            logits = model(imgs)
            loss = criterion(logits, masks, skels)

        print(f"   ✓ Loss computation successful")
        print(f"   - Batch shapes: img={imgs.shape}, mask={masks.shape}, skel={skels.shape}")
        print(f"   - Loss value: {loss.item():.4f}")
        print(f"   - Loss is finite: {torch.isfinite(loss).item()}")

        del imgs, masks, skels, logits, loss
        torch.cuda.empty_cache()
        break

except Exception as e:
    print(f"   ✗ Loss computation failed: {e}")
    import traceback
    traceback.print_exc()

# Final summary
print("\n" + "="*70)
print("VERIFICATION SUMMARY")
print("="*70)

checks_passed = [
    class_weights.max() <= 10,  # Reasonable weights
    len(train_loader) > 0,       # Has training data
    GAMMA_SKEL >= 1.5,           # High skeleton weight
    NUM_DILATIONS >= 3,          # Enough dilations
]

print(f"\nChecks passed: {sum(checks_passed)}/{len(checks_passed)}")

if all(checks_passed):
    print("\n✅ ALL CHECKS PASSED - READY TO TRAIN!")
    print("\nYou can now run:")
    print("   history = fit_with_skeleton(...)")
else:
    print("\n⚠ SOME CHECKS FAILED - Review settings above")
    if GAMMA_SKEL < 1.5:
        print("   → Increase GAMMA_SKEL to 2.0 for vessels")
    if NUM_DILATIONS < 3:
        print("   → Increase NUM_DILATIONS to 3 for vessels")

print("="*70)

Pre-Training Verification

1️⃣ Class Weights:
   ✓ Background weight: 0.10
   ✓ Mean foreground: 0.59
   ✓ Max (rare class): 10.00
   Status: ✓ GOOD

2️⃣ Testing Dataset Loading...
   ✓ Sample loaded successfully
   - Image shape: torch.Size([1, 64, 128, 128])
   - Mask shape: torch.Size([64, 128, 128])
   - Skeleton shape: torch.Size([64, 128, 128])
   - Foreground: 16418 voxels
   - Skeleton: 5279 voxels
   - Coverage: 32.2%
   Status: ✓ Skeleton generation WORKING

3️⃣ Loss Configuration:
   - ALPHA_CE (CrossEntropy): 1.0
   - BETA_DICE (Dice): 1.0
   - GAMMA_SKEL (Skeleton): 0.5 ⚠ LOW
   - NUM_DILATIONS: 3 ✓ GOOD

4️⃣ Model Status:
   - Parameters: 23,677,091
   - Device: cuda:0
   - Training mode: True

5️⃣ Data Loaders:
   - Training batches: 160
   - Validation batches: 20
   - Batch size: 2
   - Accumulation steps: 2
   - Effective batch: 4

6️⃣ Testing Forward Pass...
   ✓ Forward pass successful
   - Input: torch.Size([1, 1, 32, 64, 64])
   - Output: torch.Size([1, 41, 32, 64

In [ ]:
#@title Train Model

print("\n" + "="*60)
print("Configuration Summary")
print("="*60)
print(f"Patch size: {PATCH_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation steps: {ACCUM_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * ACCUM_STEPS}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate (max): {LR_MAX}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Loss weights: CE={ALPHA_CE}, Dice={BETA_DICE}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print("="*60)

history = fit_with_skeleton(
    model, train_loader, val_loader,
    epochs=EPOCHS,
    lr=LR_MAX,
    weight_decay=WEIGHT_DECAY,
    num_classes=NUM_CLASSES,
    save_path=CHECKPOINT_PATH,
    accum_steps=ACCUM_STEPS,
    alpha_ce=ALPHA_CE,
    beta_dice=BETA_DICE,
    gamma_skel=GAMMA_SKEL
)

print("\n⚠ Training is commented out by default.")
print("Uncomment the 'history = fit(...)' block above to start training.")


Configuration Summary
Patch size: (64, 128, 128)
Batch size: 2
Gradient accumulation steps: 2
Effective batch size: 4
Epochs: 80
Learning rate (max): 0.001
Weight decay: 1e-05
Loss weights: CE=1.0, Dice=1.0
Checkpoint path: /content/drive/MyDrive/3D_CAD_Unet/Skeleton_loss/Model/best_CADUNet3D.pt


/tmp/ipython-input-2382827413.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



Training with Skeleton Recall Loss
Loss weights: CE=1.0, Dice=1.0, Skeleton=0.5


Training:   0%|          | 0/160 [00:00<?, ?it/s]/tmp/ipython-input-2382827413.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


RuntimeError: DataLoader worker (pid 6694) is killed by signal: Killed. 

In [ ]:
#@title Plot Training History (Run after training)

def plot_training_history(history, save_path=None):
    """Plot training loss and validation dice over epochs."""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

    epochs = range(1, len(history['train_loss']) + 1)

    # Training loss
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss')
    ax1.set_title('Training Loss')
    ax1.grid(True, alpha=0.3)

    # Validation Dice
    ax2.plot(epochs, history['val_dice'], 'g-', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation Dice')
    ax2.set_title('Validation Dice Score')
    ax2.grid(True, alpha=0.3)

    # Learning rate
    ax3.plot(epochs, history['lr'], 'r-', linewidth=2)
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Learning Rate')
    ax3.set_title('Learning Rate (OneCycle)')
    ax3.set_yscale('log')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Saved plot to {save_path}")

    plt.show()

# Uncomment after training:
# plot_training_history(history, save_path=os.path.join(RESULTS_DIR, "training_history.png"))

In [ ]:
#@title Inference on New Volume

def predict_volume(img_path, model_path, output_path,
                   patch_size=PATCH_SIZE, overlap=(0.5, 0.5, 0.5)):
    """Complete inference pipeline for a single volume."""
    print("\n" + "="*60)
    print("Running Inference")
    print("="*60)
    print(f"Input: {img_path}")
    print(f"Model: {model_path}")
    print(f"Output: {output_path}")

    # Load volume
    print("\n1. Loading volume...")
    vol = nib.load(img_path).get_fdata().astype(np.float32)
    print(f"   Volume shape: {vol.shape}")

    # Normalize
    print("2. Normalizing...")
    vol = (vol - vol.mean()) / (vol.std() + 1e-6)

    # Load model
    print("3. Loading model...")
    model = CADUNet3D(in_ch=1, base=BASE_CH, num_classes=NUM_CLASSES).to(device)
    checkpoint = torch.load(model_path, map_location=device)

    # Fix for DataParallel checkpoints
    state_dict = checkpoint['model_state_dict']
    if any(k.startswith('module.') for k in state_dict.keys()):
        print("   Detected 'module.' prefix in checkpoint → cleaning keys...")
        new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(state_dict)

    model.eval()
    print(f"   Loaded checkpoint from epoch {checkpoint.get('epoch', 'unknown')}")
    print(f"   Validation Dice: {checkpoint.get('val_dice', 'unknown')}")

    # Predict
    print("4. Running sliding window prediction...")
    logits_vol = sliding_window_predict(
        vol, model,
        patch_size=patch_size,
        overlap=overlap,
        num_classes=NUM_CLASSES
    )

    # Get prediction
    print("5. Computing final segmentation...")
    pred_mask = np.argmax(logits_vol, axis=0).astype(np.int16)

    # Count labels
    unique, counts = np.unique(pred_mask, return_counts=True)
    print(f"   Predicted labels: {len(unique)}")
    for label, count in zip(unique, counts):
        pct = 100 * count / pred_mask.size
        if pct > 0.1:  # Only show labels with >0.1% presence
            print(f"     Label {label}: {count} voxels ({pct:.2f}%)")

    # Save
    print("6. Saving result...")
    save_nifti(pred_mask, img_path, output_path)

    print("\n" + "="*60)
    print("✓ Inference Complete!")
    print("="*60)

    return pred_mask

# Example usage (uncomment and modify paths after training):
test_img = "/content/drive/MyDrive/Colab Notebooks/nnUNet_raw/Dataset501_TopBrain_Segmentation/imagesTr_topbrain_ct/topcow_ct_001_0000.nii.gz"
output_pred = os.path.join(RESULTS_DIR, "prediction.nii.gz")
pred = predict_volume(test_img, CHECKPOINT_PATH, output_pred)


Running Inference
Input: /content/drive/MyDrive/Colab Notebooks/nnUNet_raw/Dataset501_TopBrain_Segmentation/imagesTr_topbrain_ct/topcow_ct_001_0000.nii.gz
Model: /content/drive/MyDrive/3D_CAD_Unet/Skeleton_loss/Model/best_CADUNet3D_SL.pt
Output: /content/drive/MyDrive/3D_CAD_Unet/Skeleton_loss/Results/prediction.nii.gz

1. Loading volume...
   Volume shape: (284, 327, 243)
2. Normalizing...
3. Loading model...
   Loaded checkpoint from epoch 32
   Validation Dice: 0.07224464416503906
4. Running sliding window prediction...


Predicting:   0%|          | 0/216 [00:00<?, ?it/s]/tmp/ipython-input-2363441414.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Predicting:  26%|██▌       | 56/216 [00:28<01:21,  1.96it/s]


5. Computing final segmentation...
   Predicted labels: 12
     Label 0: 22538800 voxels (99.88%)
6. Saving result...
✓ Saved: /content/drive/MyDrive/3D_CAD_Unet/Skeleton_loss/Results/prediction.nii.gz

✓ Inference Complete!


In [ ]:
#@title Visualize Predictions

def visualize_prediction(img_path, mask_path, pred_path=None, slice_idx=None, save_path=None):
    """Visualize image, ground truth mask, and prediction."""
    img = nib.load(img_path).get_fdata()
    mask = nib.load(mask_path).get_fdata()

    if slice_idx is None:
        slice_idx = img.shape[0] // 2  # Middle slice

    fig, axes = plt.subplots(1, 3 if pred_path else 2, figsize=(15, 5))

    # Image
    axes[0].imshow(img[slice_idx], cmap='gray')
    axes[0].set_title(f'Image (slice {slice_idx})')
    axes[0].axis('off')

    # Ground truth
    axes[1].imshow(mask[slice_idx], cmap='nipy_spectral', vmin=0, vmax=NUM_CLASSES-1)
    axes[1].set_title('Ground Truth')
    axes[1].axis('off')

    # Prediction
    if pred_path:
        pred = nib.load(pred_path).get_fdata()
        axes[2].imshow(pred[slice_idx], cmap='nipy_spectral', vmin=0, vmax=NUM_CLASSES-1)
        axes[2].set_title('Prediction')
        axes[2].axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Saved visualization to {save_path}")

    plt.show()

# Example usage:
visualize_prediction(
    img_path=test_img,
    mask_path=test_mask,
    pred_path=output_pred,
    slice_idx=32,
    save_path=os.path.join(RESULTS_DIR, "visualization.png")
)

NameError: name 'test_mask' is not defined

In [ ]:
#@title Compute Metrics

@torch.no_grad()
def compute_detailed_metrics(pred_path, gt_path, num_classes=NUM_CLASSES):
    """Compute detailed per-class metrics."""
    pred = nib.load(pred_path).get_fdata().astype(np.int64)
    gt = nib.load(gt_path).get_fdata().astype(np.int64)

    print("\n" + "="*60)
    print("Per-Class Metrics")
    print("="*60)
    print(f"{'Class':<6} {'Dice':<8} {'IoU':<8} {'Precision':<10} {'Recall':<8}")
    print("-" * 60)

    dices = []
    ious = []

    for c in range(num_classes):
        pred_c = (pred == c)
        gt_c = (gt == c)

        tp = np.sum(pred_c & gt_c)
        fp = np.sum(pred_c & ~gt_c)
        fn = np.sum(~pred_c & gt_c)

        if tp + fp + fn == 0:
            continue

        dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        dices.append(dice)
        ious.append(iou)

        print(f"{c:<6} {dice:<8.4f} {iou:<8.4f} {precision:<10.4f} {recall:<8.4f}")

    print("-" * 60)
    print(f"{'Mean':<6} {np.mean(dices):<8.4f} {np.mean(ious):<8.4f}")
    print("=" * 60)

    return {
        'mean_dice': np.mean(dices),
        'mean_iou': np.mean(ious),
        'per_class_dice': dices,
        'per_class_iou': ious
    }

# Example usage:
# metrics = compute_detailed_metrics(output_pred, test_mask)

In [ ]:
#@title Batch Inference

def batch_inference(img_dir, output_dir, model_path, img_pattern="*.nii.gz"):
    """Run inference on all images in a directory."""
    os.makedirs(output_dir, exist_ok=True)

    img_files = sorted(glob.glob(os.path.join(img_dir, img_pattern)))
    print(f"\nFound {len(img_files)} images for inference")

    for img_path in img_files:
        basename = os.path.basename(img_path)
        output_path = os.path.join(output_dir, basename.replace('.nii.gz', '_pred.nii.gz'))

        try:
            predict_volume(img_path, model_path, output_path)
        except Exception as e:
            print(f"✗ Error processing {basename}: {e}")
            continue

    print(f"\n✓ Batch inference complete. Results saved to {output_dir}")

# Example usage:
# batch_inference(
#     img_dir="/content/drive/MyDrive/test_images/",
#     output_dir=os.path.join(RESULTS_DIR, "batch_predictions"),
#     model_path=CHECKPOINT_PATH
# )

In [ ]:
#@title Save Configuration

def save_config(save_path=None):
    """Save training configuration to file."""
    if save_path is None:
        save_path = os.path.join(RESULTS_DIR, "config.txt")

    config = f"""CAD U-Net 3D Training Configuration
{'='*60}
Data:
  DATA_DIR: {DATA_DIR}
  IMG_GLOB: {IMG_GLOB}
  MSK_GLOB: {MSK_GLOB}
  LABELMAP_TXT: {LABELMAP_TXT}
  NUM_CLASSES: {NUM_CLASSES}

Model:
  Architecture: CAD U-Net 3D
  BASE_CH: {BASE_CH}
  PATCH_SIZE: {PATCH_SIZE}

Training:
  BATCH_SIZE: {BATCH_SIZE}
  ACCUM_STEPS: {ACCUM_STEPS}
  Effective batch size: {BATCH_SIZE * ACCUM_STEPS}
  EPOCHS: {EPOCHS}
  LR_MAX: {LR_MAX}
  WEIGHT_DECAY: {WEIGHT_DECAY}

Loss:
  ALPHA_CE: {ALPHA_CE}
  BETA_DICE: {BETA_DICE}

Other:
  FOREGROUND_SAMPLING_PROB: {FOREGROUND_SAMPLING_PROB}
  TRAIN_VAL_SPLIT: {TRAIN_VAL_SPLIT}
  RANDOM_SEED: {RANDOM_SEED}
  RESULTS_DIR: {RESULTS_DIR}
  CHECKPOINT_PATH: {CHECKPOINT_PATH}

Device: {device}
{'='*60}
"""

    with open(save_path, 'w') as f:
        f.write(config)

    print(f"✓ Configuration saved to {save_path}")
    return config

# Save configuration
config_text = save_config()
print("\n" + config_text)

# ============================================================================
print("\n" + "="*60)
print("✓ All components loaded successfully!")
print("="*60)
print("\nNext steps:")
print("1. Verify your data paths are correct")
print("2. Uncomment the training code in the 'Train Model' section")
print("3. Run training: history = fit(...)")
print("4. Monitor training progress")
print("5. Use inference functions to predict on new volumes")
print("\nFor questions or issues, check the configuration summary above.")
print("="*60)